# 01 · Calidad de datos
**Pregunta:** ¿qué registros permiten analizar el comportamiento de un cliente?

Se inspecciona el original de UCI y se reconcilian las reglas de limpieza. Ejecuta primero
`python -m customer_segmentation.pipeline` desde la raíz. El proceso descarga el original
y crea los archivos que utilizan los cinco notebooks.

Fuente: [Online Retail II · UCI](https://doi.org/10.24432/C5CG6D), Chen (2012), CC BY 4.0.
Una fila representa una línea de producto, no un pedido.

In [1]:
import json
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "config/settings.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Abre el notebook dentro del repositorio.")
    ROOT = ROOT.parent
config = yaml.safe_load((ROOT / "config/settings.yaml").read_text(encoding="utf-8"))
REPORTS = ROOT / "reports"

In [2]:
from customer_segmentation.cleaning import read_transactions

raw = read_transactions(ROOT / config["raw_file"])
display(raw.head())
display(
    raw.groupby("source_sheet").agg(
        lineas=("Invoice", "size"),
        primera_fecha=("InvoiceDate", "min"),
        ultima_fecha=("InvoiceDate", "max"),
    )
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


,lineas,primera_fecha,ultima_fecha
source_sheet,,,
Year 2009-2010,525461,2009-12-01 07:45:00,2010-12-09 20:01:00
Year 2010-2011,541910,2010-12-01 08:26:00,2011-12-09 12:50:00


## Faltantes y solapamiento
Las hojas se solapan en diciembre de 2010. Eliminar todas las líneas idénticas también borraría
repeticiones dentro de una misma factura. Se eliminan únicamente copias entre hojas, conservando
la multiplicidad máxima observada. Las repeticiones internas se informan y conservan.

In [3]:
missing = raw.isna().sum().rename("Faltantes originales")
display(missing.to_frame())
quality = json.loads((REPORTS / "quality_report.json").read_text(encoding="utf-8"))
reasons = [
    "cross_sheet_copies",
    "missing_customer",
    "invalid_date",
    "invalid_invoice",
    "invalid_quantity",
    "invalid_price",
    "inconsistent_cancellation",
]
exclusions = pd.Series({name: quality[name] for name in reasons}, name="Líneas excluidas")
display(exclusions.to_frame())
assert exclusions.sum() + quality["retained_rows"] == len(raw)
print(f"Original: {len(raw):,} · Conservadas: {quality['retained_rows']:,}")
print(f"Repeticiones idénticas internas conservadas: {quality['identical_lines_retained']:,}")

Original: 1,067,371 · Conservadas: 809,491
Repeticiones idénticas internas conservadas: 11,812


,Faltantes originales
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0
source_sheet,0


,Líneas excluidas
cross_sheet_copies,22523
missing_customer,235287
invalid_date,0
invalid_invoice,0
invalid_quantity,0
invalid_price,70
inconsistent_cancellation,0


## Compras y cancelaciones
Las exclusiones son secuenciales: un registro se cuenta una sola vez. La cantidad de IDs ausentes
tras resolver el solapamiento es distinta del faltante bruto. Las cancelaciones válidas se conservan
para su análisis, pero no aumentan frecuencia ni gasto bruto RFM.

In [4]:
clean = pd.read_parquet(ROOT / "data/processed/transactions.parquet")
display(clean.groupby("status").agg(lineas=("invoice", "size"), importe=("amount", "sum")))
assert clean.customer_id.notna().all()
assert clean.loc[clean.status.eq("purchase"), "amount"].gt(0).all()
assert clean.loc[clean.status.eq("cancellation"), "amount"].lt(0).all()
print(f"Clientes identificados en todo el historial depurado: {clean.customer_id.nunique():,}")

Clientes identificados en todo el historial depurado: 5,939


,lineas,importe
status,,
cancellation,18446,-1.087882e+06
purchase,791045,1.743155e+07


## Decisión
Se puede construir RFM sobre clientes identificados con compras positivas. La descripción ausente
no impide hacerlo. No se inventa un cliente para las compras anónimas y no se elimina automáticamente
un gran comprador por ser extremo. El siguiente notebook restringe el análisis al año de observación.

[Continuar: análisis comercial](02_exploratory_analysis.ipynb)